# P-P plot

This notebook shows how to assess the performance of trained posteriors from their loss functions and P-P plots.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from cogwheel import gw_plotting

from cogwheel_machine import utils, training, pp_plot

In [ ]:
%matplotlib widget

## Analyze a single model

In [ ]:
modeldir = Path('../data/run_0/model_0/')
rundir = modeldir.parent

In [ ]:
training.plot_loss(modeldir)

In [ ]:
# The first time this will take a few seconds. Then the credible_intervals are saved.
credible_intervals = pp_plot.get_credible_intervals(
    modeldir, n_data=2000, n_processes=20)

pp_plot.pp_plot(credible_intervals)
plt.title(modeldir.name)

## Compare multiple models

In [ ]:
def compare_model_losses(modeldirs, x_axis='Epoch'):
    """Loss vs epoch for all models."""
    fig, ax = plt.subplots()
    colors = gw_plotting.plotting.gen_colors(len(modeldirs))
    
    for i, modeldir in enumerate(modeldirs):
        model_config = utils.load_model_config(modeldir)

        # Choose parameters to highlight in the legend:
        label = (model_config.POSTERIOR_NN_KWARGS['hidden_features'],
                 model_config.TRAIN_KWARGS['learning_rate'])

        epochs, training_loss, validation_loss = training.load_loss(modeldir)

        if x_axis.lower() == 'epoch':
            x_values = epochs
        elif x_axis.lower() == 'runtime':
            x_values = training.load_runtime(modeldir)
        else:
            raise ValueError('`x_axis` needs to be "Epoch" or "Runtime".')

        ax.plot(x_values, training_loss, color=colors[i], alpha=.33)
        ax.plot(x_values, validation_loss, '.', color=colors[i],
                label=f'{modeldir.name} {label}')
    
    # Should match the label's parameters
    legend = ax.legend(title='Hidden features, learning rate')
    ax.add_artist(legend)

    # Second legend:
    handles = [Line2D([0], [0], color='black', alpha=0.33),
               Line2D([0], [0], marker='.', color='black', linestyle='None')]
    plt.legend(handles, ['Training', 'Validation'], loc='center right')

    plt.xlabel(x_axis)
    plt.ylabel('Loss')
    plt.grid(ls=':')
    plt.autoscale(tight=True)


In [ ]:
modeldirs = sorted(
    rundir.glob('model_*/'),
    key=lambda path: int(path.name.removeprefix('model_')))

In [ ]:
# Loss vs epoch
compare_model_losses(modeldirs, x_axis='Epoch')

In [ ]:
# Loss vs runtime
compare_model_losses(modeldirs, x_axis='Runtime')